# Federated batch correction with FedRBE

FedRBE (`flimmaBatchCorrection` FeatureCloud app) is the federated equivalent of `limma::removeBatchEffect` (Ritchie et al. 2015). This notebook prepares FedRBE-style client folders for the four-client Quartet multiomics federation defined in `02_prepare_RBE_inputs.ipynb` (`client_01_L01`, `client_02_L02`, `client_03_L05_L04`, `client_04_L03_L14`) and runs an in-process simulation of the federated correction.

Each client carries 12 or 24 libraries from one (or two) source labs; the `batch_col` inside each client is the original Quartet `batch`. The reference donor is `D6`. The reference batch (alphabetically last batch of the last client) is hard-coded by `prepare_fedrbe_clients` so the federated and central corrections produce comparable contrasts.


In [ ]:
from pathlib import Path
import sys

WORKDIR = Path.cwd() if Path("figshare_data").exists() else Path("evaluation_data/multiomics")
REPO_ROOT = WORKDIR.resolve().parents[1]
if str(WORKDIR.resolve()) not in sys.path:
    sys.path.insert(0, str(WORKDIR.resolve()))

from fedrbe_multiomics_utils import MODALITIES, run_all_fedsim

print(f"WORKDIR: {WORKDIR.resolve()}")
print(f"Modalities: {MODALITIES}")


## Local FedRBE-equivalent correction

`run_all_fedsim` writes per-client folders (`intensities_log_UNION.tsv`, `design.tsv`, `config.yml`) under `before/<modality>/<client>/` using the same conventions as the other FedRBE datasets, then runs the same XTX/XTY aggregation and batch-effect subtraction as the FeatureCloud app. The original Quartet `batch` is preserved as `batch_col` inside each client. Lab → client mapping comes from `CLIENT_LAB_MAPS` in `fedrbe_multiomics_utils.py`. This local path makes the correction reproducible without the FeatureCloud / Docker stack; the optional cell at the bottom of the notebook runs the real FeatureCloud app instead.


In [ ]:
correction_summary = run_all_fedsim(WORKDIR)
correction_summary

## Output files

Per-modality simulated FedRBE outputs are written beside the central limma outputs. The combined all-modality matrix uses the same equal-weight block construction as the central correction notebook.

In [ ]:
for modality in MODALITIES:
    out_dir = WORKDIR / "after" / modality
    fed_path = out_dir / "FedSim_corrected_data.tsv"
    print(f"{modality}: FedSim_corrected_data.tsv={fed_path.exists()}")

print(f"Combined FedSim k-means matrix: {(WORKDIR / 'after' / 'all_modalities_fedsim_kmeans_matrix.tsv').exists()}")
print(f"Client groups: {(WORKDIR / 'before' / 'fedrbe_client_groups.tsv').exists()}")
print(f"Summary: {(WORKDIR / 'after' / 'fedsim_correction_summary.tsv').exists()}")


## Optional FeatureCloud app run

Set `RUN_FEATURECLOUD = True` only in an environment with the FeatureCloud Python package, Docker Python SDK, PyYAML, and a running Docker daemon. The local `featurecloud.ai/bcorrect:latest` image is required.

In [ ]:
RUN_FEATURECLOUD = False

if RUN_FEATURECLOUD:
    import pandas as pd
    from copy import deepcopy

    sys.path.insert(0, str(REPO_ROOT))
    from evaluation_utils import featurecloud_api_extension as util
    from evaluation_utils.featurecloud_api_extension import postprocess_results

    app_image_name = "featurecloud.ai/bcorrect:latest"
    base_config = {
        "flimmaBatchCorrection": {
            "data_filename": "intensities_log_UNION.tsv",
            "design_filename": "design.tsv",
            "expression_file_flag": True,
            "index_col": "rowname",
            "batch_col": "batch",
            "covariates": ["D5", "F7", "M8"],
            "separator": "\t",
            "design_separator": "\t",
            "normalizationMethod": None,
            "smpc": False,
            "min_samples": 0,
            "position": None,
            "reference_batch": False,
        }
    }

    for modality in MODALITIES:
        all_groups = pd.read_csv(WORKDIR / "before" / "fedrbe_client_groups.tsv", sep="\t")
        groups = all_groups[all_groups["modality"] == modality].reset_index(drop=True)
        clients = [str(WORKDIR / "before" / modality / client) for client in groups["client"]]
        config_changes = []
        for _, row in groups.iterrows():
            reference = row["reference_batch"] if isinstance(row["reference_batch"], str) and row["reference_batch"] != "False" else False
            config_changes.append({
                "flimmaBatchCorrection": {
                    "position": int(row["position"]),
                    "reference_batch": reference,
                    "smpc": False,
                }
            })
        experiment = util.Experiment(
            name=f"Multiomics {modality}",
            fc_data_dir=str(WORKDIR),
            clients=clients,
            app_image_name=app_image_name,
            config_files=[deepcopy(base_config) for _ in clients],
            config_file_changes=config_changes,
        )
        result_files_zipped, _, _ = experiment.run_test()
        result_path = WORKDIR / "after" / modality / "FedApp_corrected_data.tsv"
        result_path.parent.mkdir(parents=True, exist_ok=True)
        postprocess_results(experiment, result_files_zipped, str(result_path))
else:
    print("FeatureCloud run skipped. Local FedSim outputs above are ready for downstream checks.")